# Assignment 1.1

1. One-Hot Encoding
2. TF-IDF
3. Word2Vec
4. BERT Contextual Embeddings


In [2]:
pip install -q numpy pandas scikit-learn gensim transformers torch

In [3]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


## 1. One-Hot Encoding

In [4]:
sentence = "he started driving when he was 22 years old"

tokens = sentence.lower().split()
vocab = sorted(set(tokens))
word_to_idx = {word: i for i, word in enumerate(vocab)}

one_hot = np.zeros((len(tokens), len(vocab)), dtype=int)

for row, token in enumerate(tokens):
    one_hot[row, word_to_idx[token]] = 1

one_hot_df = pd.DataFrame(one_hot, index=tokens, columns=vocab)
one_hot_df


,22,driving,he,old,started,was,when,years
he,0,0,1,0,0,0,0,0
started,0,0,0,0,1,0,0,0
driving,0,1,0,0,0,0,0,0
when,0,0,0,0,0,0,1,0
he,0,0,1,0,0,0,0,0
was,0,0,0,0,0,1,0,0
22,1,0,0,0,0,0,0,0
years,0,0,0,0,0,0,0,1
old,0,0,0,1,0,0,0,0


In [5]:
print("Vocabulary:", vocab)
print("Matrix shape:", one_hot_df.shape)


Vocabulary: ['22', 'driving', 'he', 'old', 'started', 'was', 'when', 'years']
Matrix shape: (9, 8)


Each token is represented by a sparse vector whose length is equal to the vocabulary size. Only the position corresponding to that token is 1; all other positions are 0.

## 2. TF-IDF

In [6]:
documents = [
    "machine learning models learn patterns from data",
    "deep learning uses neural networks for learning",
    "natural language processing works with text data",
    "word embeddings represent words as numerical vectors"
]

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(documents)

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf.get_feature_names_out(),
    index=[f"Document {i+1}" for i in range(len(documents))]
)

tfidf_df.round(3)


,as,data,deep,embeddings,for,from,language,learn,learning,machine,...,patterns,processing,represent,text,uses,vectors,with,word,words,works
Document 1,0.000,0.316,0.000,0.000,0.000,0.4,0.000,0.4,0.316,0.4,...,0.4,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
Document 2,0.000,0.000,0.365,0.000,0.365,0.0,0.000,0.0,0.576,0.0,...,0.0,0.000,0.000,0.000,0.365,0.000,0.000,0.000,0.000,0.000
Document 3,0.000,0.306,0.000,0.000,0.000,0.0,0.389,0.0,0.000,0.0,...,0.0,0.389,0.000,0.389,0.000,0.000,0.389,0.000,0.000,0.389
Document 4,0.378,0.000,0.000,0.378,0.000,0.0,0.000,0.0,0.000,0.0,...,0.0,0.000,0.378,0.000,0.000,0.378,0.000,0.378,0.378,0.000


In [7]:
similarity_matrix = cosine_similarity(tfidf_matrix)

pd.DataFrame(
    similarity_matrix,
    index=tfidf_df.index,
    columns=tfidf_df.index
).round(3)


,Document 1,Document 2,Document 3,Document 4
Document 1,1.000,0.182,0.097,0.0
Document 2,0.182,1.000,0.000,0.0
Document 3,0.097,0.000,1.000,0.0
Document 4,0.000,0.000,0.000,1.0


TF-IDF gives larger weights to terms that are important in a document but less common across the corpus. Cosine similarity can then be used to compare the resulting document vectors.

## 3. Word2Vec

In [8]:
from gensim.models import Word2Vec

corpus = [
    "king queen prince princess royal family",
    "king queen palace kingdom royal",
    "man woman boy girl family",
    "doctor nurse hospital patient medicine",
    "teacher student school class education",
    "computer software hardware technology machine",
    "machine learning artificial intelligence data model",
    "deep learning neural network model data",
    "natural language processing text words language",
    "word embeddings vectors semantic meaning words"
]

tokenized_corpus = [sentence.lower().split() for sentence in corpus]

w2v_model = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=50,
    window=3,
    min_count=1,
    sg=1,
    workers=1,
    epochs=300,
    seed=42
)


In [9]:
print("Vector for 'king':")
print(w2v_model.wv["king"][:10])

print("\nMost similar words to 'king':")
print(w2v_model.wv.most_similar("king", topn=5))


Vector for 'king':
[ 0.01853335  0.00872447 -0.00447936 -0.01953284  0.00888731  0.00320114
 -0.01994289  0.01412867  0.0075176  -0.01632218]

Most similar words to 'king':
[('vectors', 0.530114471912384), ('word', 0.48694905638694763), ('nurse', 0.48206543922424316), ('medicine', 0.45810988545417786), ('language', 0.4521036744117737)]


In [10]:
pairs = [
    ("king", "queen"),
    ("king", "doctor"),
    ("learning", "model"),
    ("language", "words")
]

for word1, word2 in pairs:
    score = w2v_model.wv.similarity(word1, word2)
    print(f"{word1:10s} - {word2:10s}: {score:.3f}")


king       - queen     : 0.286
king       - doctor    : 0.032
learning   - model     : 0.635
language   - words     : 0.521


Word2Vec learns dense vectors from surrounding context. Words that occur in similar contexts tend to receive more similar vectors. Unlike one-hot and TF-IDF representations, the vector dimensions are learned rather than tied directly to vocabulary positions.

## 4. BERT Contextual Embeddings

In [11]:
import torch
from transformers import AutoTokenizer, AutoModel

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name)
bert_model.eval()


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [12]:
def get_contextual_embedding(sentence, target_word):
    encoded = tokenizer(sentence, return_tensors="pt")

    with torch.no_grad():
        output = bert_model(**encoded)

    tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
    positions = [i for i, token in enumerate(tokens) if token == target_word.lower()]

    if not positions:
        raise ValueError(f"'{target_word}' was not found as a single BERT token.")

    embedding = output.last_hidden_state[0, positions[0]]
    return tokens, embedding


sentence_1 = "I deposited money in the bank."
sentence_2 = "We sat beside the river bank."

tokens_1, bank_finance = get_contextual_embedding(sentence_1, "bank")
tokens_2, bank_river = get_contextual_embedding(sentence_2, "bank")

print(tokens_1)
print(tokens_2)


['[CLS]', 'i', 'deposited', 'money', 'in', 'the', 'bank', '.', '[SEP]']
['[CLS]', 'we', 'sat', 'beside', 'the', 'river', 'bank', '.', '[SEP]']


In [13]:
bert_similarity = torch.nn.functional.cosine_similarity(
    bank_finance.unsqueeze(0),
    bank_river.unsqueeze(0)
).item()

print(f"Cosine similarity between the two contextual 'bank' embeddings: {bert_similarity:.3f}")


Cosine similarity between the two contextual 'bank' embeddings: 0.532


In [14]:
print("First 10 values for 'bank' in the financial sentence:")
print(bank_finance[:10])

print("\nFirst 10 values for 'bank' in the river sentence:")
print(bank_river[:10])


First 10 values for 'bank' in the financial sentence:
tensor([ 0.5343, -0.3769, -0.1306,  0.1728,  0.9833,  0.1560, -0.7062,  0.8090,
        -0.1534, -0.0731])

First 10 values for 'bank' in the river sentence:
tensor([ 0.4702, -0.5839, -0.0954, -0.1607, -0.3511,  0.0802,  0.3014,  1.4108,
         0.1055, -0.5201])


BERT produces contextual embeddings. The word **bank** receives a different vector in the financial sentence and the river sentence because its representation depends on the surrounding words. This is the main difference from static embeddings such as Word2Vec, where a word has one fixed vector.